# Perch (ONNX) + SVM Inference

Runs Perch via ONNX Runtime for CPU efficiency. Uses existing SVM artifacts.

**Required Datasets:**
- Competition data
- ONNX Perch model dataset (`perch_v1.onnx`)
- SVM artifacts dataset (model, scaler, species list)

In [ ]:
import glob, subprocess, sys
try:
    import onnxruntime as ort
    print('onnxruntime preinstalled:', ort.__version__)
except ImportError:
    ORT_WHEEL_GLOB = '/kaggle/input/**/onnxruntime-*.whl'
    whls = glob.glob(ORT_WHEEL_GLOB, recursive=True)
    assert whls, f'onnxruntime not found and no wheel matched {ORT_WHEEL_GLOB}'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', whls[0]], check=True)
    import onnxruntime as ort
    print('onnxruntime installed from wheel:', ort.__version__)

onnxruntime installed from wheel: 1.24.4


In [2]:
import os, gc, time, json, warnings, joblib
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.dummy import DummyClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


In [3]:
DATA_ROOT = Path('/kaggle/input/competitions/birdclef-2026')

# Paths for model and SVM artifacts
import glob
_onnx = (sorted(glob.glob('/kaggle/input/**/*perch*dft*.onnx', recursive=True))
         or sorted(glob.glob('/kaggle/input/**/*perch*.onnx', recursive=True)))
assert _onnx, 'Attach tuckerarrants/perch-v2-no-dft-onnx'
ONNX_PATH = _onnx[0]

SVM_MODEL_PATH   = '/kaggle/input/notebooks/marialuizalaranjeira/notebook6232f7fefe/svm_perch_model.pkl'
SVM_SCALER_PATH  = '/kaggle/input/notebooks/marialuizalaranjeira/notebook6232f7fefe/svm_perch_scaler.pkl'
SVM_SPECIES_PATH = '/kaggle/input/notebooks/marialuizalaranjeira/notebook6232f7fefe/svm_perch_active_species.pkl'

SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12
BATCH_FILES    = 16

print('ONNX :', ONNX_PATH)
for p in [SVM_MODEL_PATH, SVM_SCALER_PATH, SVM_SPECIES_PATH]:
    print(f'  {p:<60} exists={Path(p).exists()}')

ONNX : /kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx
  /kaggle/input/notebooks/marialuizalaranjeira/notebook6232f7fefe/svm_perch_model.pkl exists=True
  /kaggle/input/notebooks/marialuizalaranjeira/notebook6232f7fefe/svm_perch_scaler.pkl exists=True
  /kaggle/input/notebooks/marialuizalaranjeira/notebook6232f7fefe/svm_perch_active_species.pkl exists=True


In [4]:
# Required for unpickling SafeClassifier
class SafeClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        try:
            if len(self.classes_) < 2:
                raise ValueError('only one class')
            from sklearn.base import clone
            self.model_ = clone(self.base_estimator)
            self.model_.fit(X, y)
        except Exception:
            self.model_ = DummyClassifier(strategy='constant', constant=y[0])
            self.model_.fit(X, y)
        return self
    def predict_proba(self, X):
        m = self.model_
        if hasattr(m, 'predict_proba'):
            return m.predict_proba(X)
        d = m.decision_function(X)
        p = 1.0 / (1.0 + np.exp(-d))
        return np.column_stack([1.0 - p, p])

In [ ]:
N_CPU = os.cpu_count() or 4
so = ort.SessionOptions()
so.intra_op_num_threads = N_CPU
so.inter_op_num_threads = 1
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
perch_sess = ort.InferenceSession(ONNX_PATH, sess_options=so,
                                  providers=['CPUExecutionProvider'])

ONNX_IN  = perch_sess.get_inputs()[0].name
out_names = [o.name for o in perch_sess.get_outputs()]
ONNX_OUT = 'embedding' if 'embedding' in out_names else out_names[0]
print(f'input={ONNX_IN!r}  embed_output={ONNX_OUT!r}  all_outputs={out_names}')

def perch_embed(windows_batch):
    """(N,160000) float32 -> (N,1536) raw embeddings (un-normalized).
    StandardScaler is applied later, exactly as in training."""
    out = perch_sess.run([ONNX_OUT], {ONNX_IN: windows_batch.astype(np.float32)})[0]
    return out.astype(np.float32, copy=False)

input='inputs'  embed_output='embedding'  all_outputs=['embedding', 'spatial_embedding', 'spectrogram', 'label']


In [6]:
svm_model      = joblib.load(SVM_MODEL_PATH)
scaler         = joblib.load(SVM_SCALER_PATH)
ACTIVE_SPECIES = joblib.load(SVM_SPECIES_PATH)
print(f'SVM loaded. Active species: {len(ACTIVE_SPECIES)}')

sample_sub  = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
ALL_SPECIES = sample_sub.columns[1:].tolist()
N_SPECIES   = len(ALL_SPECIES)
MISSING_PROB = 1.0 / N_SPECIES
UNIFORM     = np.full(N_SPECIES, MISSING_PROB, dtype=np.float32)

active_to_sub_idx = np.array([
    ALL_SPECIES.index(sp) if sp in ALL_SPECIES else -1
    for sp in ACTIVE_SPECIES
], dtype=np.int64)
print(f'Active species not in submission: {(active_to_sub_idx < 0).sum()} (should be 0)')

SVM loaded. Active species: 185
Active species not in submission: 0 (should be 0)


In [7]:
def read_soundscape_60s(path):
    try:
        y, sr = sf.read(str(path), dtype='float32', always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr != SR:
            import librosa
            y = librosa.resample(y, orig_sr=sr, target_sr=SR).astype(np.float32)
        if len(y) < FILE_SAMPLES:
            y = np.pad(y, (0, FILE_SAMPLES - len(y)))
        elif len(y) > FILE_SAMPLES:
            y = y[:FILE_SAMPLES]
        return y
    except Exception as e:
        print(f'load failed {path}: {e}')
        return np.zeros(FILE_SAMPLES, dtype=np.float32)

In [8]:
rows_by_stem = {}
for row_id in sample_sub['row_id']:
    parts = row_id.rsplit('_', 1)
    if len(parts) == 2 and parts[1].isdigit():
        rows_by_stem.setdefault(parts[0], []).append((row_id, int(parts[1])))
stems = sorted(rows_by_stem.keys())
print(f'Soundscapes: {len(stems)}')

results = {}
audio_dir_test  = DATA_ROOT / 'test_soundscapes'
audio_dir_train = DATA_ROOT / 'train_soundscapes'
t_audio = t_perch = t_svm = 0.0
t0 = time.time()

for bs in tqdm(range(0, len(stems), BATCH_FILES), desc='Perch(ONNX) batches'):
    batch_stems = stems[bs:bs + BATCH_FILES]
    bw = np.zeros((len(batch_stems) * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
    valid = np.ones(len(batch_stems) * N_WINDOWS, dtype=bool)

    ta = time.time()
    for i, stem in enumerate(batch_stems):
        cand = audio_dir_test / f'{stem}.ogg'
        if not cand.exists():
            cand = audio_dir_train / f'{stem}.ogg'
        y = read_soundscape_60s(cand)
        if np.max(np.abs(y)) == 0:
            valid[i*N_WINDOWS:(i+1)*N_WINDOWS] = False
        bw[i*N_WINDOWS:(i+1)*N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    t_audio += time.time() - ta

    tp = time.time()
    emb = perch_embed(bw)
    t_perch += time.time() - tp

    ts = time.time()
    emb = scaler.transform(emb)
    probs = svm_model.predict_proba(emb)
    t_svm += time.time() - ts

    for i, stem in enumerate(batch_stems):
        for row_id, end_time in rows_by_stem[stem]:
            w = end_time // WINDOW_SEC - 1
            gr = i * N_WINDOWS + w
            if not valid[gr]:
                results[row_id] = UNIFORM.copy(); continue
            row = np.full(N_SPECIES, MISSING_PROB, dtype=np.float32)
            row[active_to_sub_idx] = probs[gr]
            results[row_id] = row

    del bw, emb, probs
    gc.collect()

total = time.time() - t0
print(f'\nTotal {total/60:.1f} min | audio {t_audio/60:.1f} | '
      f'Perch {t_perch/60:.1f} | SVM {t_svm/60:.1f}')
print(f'Rows: {len(results)} / {len(sample_sub)}')

Soundscapes: 1


Perch(ONNX) batches:   0%|          | 0/1 [00:00<?, ?it/s]

load failed /kaggle/input/competitions/birdclef-2026/train_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg: Error opening '/kaggle/input/competitions/birdclef-2026/train_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg': System error.

Total 0.0 min | audio 0.0 | Perch 0.0 | SVM 0.0
Rows: 3 / 3


In [9]:
for rid in sample_sub['row_id']:
    results.setdefault(rid, UNIFORM.copy())

prob_matrix = np.vstack([results[rid] for rid in sample_sub['row_id']])
submission  = pd.DataFrame(prob_matrix, columns=ALL_SPECIES)
submission.insert(0, 'row_id', sample_sub['row_id'].values)

assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)
assert not submission.isnull().any().any()
assert (submission.iloc[:, 1:].values >= 0).all() and (submission.iloc[:, 1:].values <= 1).all()

submission.to_csv('submission.csv', index=False)
print('submission.csv', submission.shape)
submission.head(3)

submission.csv (3, 235)


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
